In [ ]:
                                      # Build & Deploy an AI Resume Analyzer (Azure AI Foundry)

In [1]:
pip install azure-ai-projects azure-core azure-storage-blob


   ---------------------------------------- 0/7 [isodate]
   ----- ---------------------------------- 1/7 [azure-core]
   ----- ---------------------------------- 1/7 [azure-core]
   ----- ---------------------------------- 1/7 [azure-core]
   ----- ---------------------------------- 1/7 [azure-core]
   ----- ---------------------------------- 1/7 [azure-core]
   ----- ---------------------------------- 1/7 [azure-core]
   ----- ---------------------------------- 1/7 [azure-core]
   ----- ---------------------------------- 1/7 [azure-core]
   ----------- ---------------------------- 2/7 [azure-storage-blob]
   ----------- ---------------------------- 2/7 [azure-storage-blob]
   ----------- ---------------------------- 2/7 [azure-storage-blob]
   ----------- ---------------------------- 2/7 [azure-storage-blob]
   ----------- ---------------------------- 2/7 [azure-storage-blob]
   ----------- ---------------------------- 2/7 [azure-storage-blob]
   ----------- ------------------------

In [2]:
pip install --upgrade openai azure-ai-projects azure-identity

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   --------------------------- ------------ 0.8/1.2 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 2.2 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.29.0
    Uninstalling openai-2.29.0:
      Successfully uninstalled openai-2.29.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Build & Deploy an AI Resume Analyzer 

In [44]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("key")

client = OpenAI(
    api_key=api_key, 
    base_url="https://him123.openai.azure.com/openai/v1"
)

In [45]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
resume_data = """
RICHARD SANCHEZ
Education: Master of Business Management (2029-2030)
Work: Marketing Manager at Borcelle Studio (2030-Present)
Skills: Project Management, PR, Leadership...
"""

# 2. Use the Responses API for structured extraction
# Use the structured input pattern for GPT-4.1
# Updated call with 2026-compliant type naming
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": "Extract the following details into a JSON object: Name, Education, Latest Job, and Top 3 Skills. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",  # Changed from 'text'
                        # "text": f"RESUME SOURCE:\n{resume_text}"
                        "text": f"RESUME SOURCE:\n{resume_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Name": "Richard Sanchez",\n  "Education": "Master of Business Management (2029-2030)",\n  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",\n  "Top 3 Skills": [\n    "Project Management",\n    "PR",\n    "Leadership"\n  ]\n}', type='output_text', logprobs=[])]


In [46]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Name": "Richard Sanchez",
  "Education": "Master of Business Management (2029-2030)",
  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",
  "Top 3 Skills": [
    "Project Management",
    "PR",
    "Leadership"
  ]
}


In [47]:
from azure.storage.blob import BlobServiceClient

AZURE_STORAGE_CONNECTION_STRING = os.getenv("endpoints")
blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)
# container_name = "resume-output"
container_name = "himanicont"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print(" Upload successful")

 Upload successful


In [48]:
import datetime
file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [49]:
blob_client = blob_service_client.get_blob_client(
    container="himanicont",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F7730018D6E"',
 'last_modified': datetime.datetime(2026, 4, 21, 7, 25, 38, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xf2\x08\x1d\xecq\x8f\xaa\xe1\xc3\x97\x19\x16\xbd\x81%;'),
 'client_request_id': '4b23b737-3d53-11f1-a00d-db7f0d5fbf22',
 'request_id': 'd87337b8-e01e-0004-7160-d16395000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 7, 25, 38, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [50]:
print(f"Stored as: {file_name}")

Stored as: resume_20260421_125536.txt


In [51]:
# Build & Deploy an AI Research Paper Summarizer (Azure AI Foundry)
# Objective
# Students will:
# Create AI resource
# Build an AI agent
# Deploy it
# Call it using Python
# Store summarized output in cloud

In [52]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("key")

client = OpenAI(
    api_key=api_key, 
    base_url="https://him123.openai.azure.com/openai/v1"
)

In [53]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
paper_data = """
Title: Artificial Intelligence in Healthcare

This research paper explains how AI is transforming healthcare systems.
It discusses machine learning models used for disease prediction,
early diagnosis, and drug discovery.
"""

# 2. Use the Responses API for structured extraction
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Summarize the following research paper into a JSON object with: Title, Short Summary, Detailed Summary, Key Contributions (3 points), Methodology, Conclusion, and Keywords. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",
                        "text": f"PAPER SOURCE:\n{paper_data}"
                    }
                ]
            }
        ]
    )

    print("--- SUMMARIZED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Summarization Error: {e}")

--- SUMMARIZED DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Title": "Artificial Intelligence in Healthcare",\n  "Short Summary": "The paper explores how artificial intelligence (AI) is revolutionizing healthcare through various applications such as disease prediction, early diagnosis, and drug discovery.",\n  "Detailed Summary": "This research paper examines the growing role of AI in healthcare, highlighting specific ways machine learning models and advanced algorithms are being integrated into medical practice. The discussion covers how AI systems enhance disease prediction accuracy, enable early detection of health conditions, and accelerate the drug discovery process. It also addresses the implications of these technologies for clinical workflow, patient outcomes, and operational efficiency within healthcare institutions.",\n  "Key Contributions": [\n    "Analysis of machine learning models for improved disease prediction.",\n    "Evaluation of AI tools facilitating earl

In [54]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    summary_data = json.loads(raw_json_string)

    print("--- SUCCESS: SUMMARY PARSED ---")
    print(json.dumps(summary_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: SUMMARY PARSED ---
{
  "Title": "Artificial Intelligence in Healthcare",
  "Short Summary": "The paper explores how artificial intelligence (AI) is revolutionizing healthcare through various applications such as disease prediction, early diagnosis, and drug discovery.",
  "Detailed Summary": "This research paper examines the growing role of AI in healthcare, highlighting specific ways machine learning models and advanced algorithms are being integrated into medical practice. The discussion covers how AI systems enhance disease prediction accuracy, enable early detection of health conditions, and accelerate the drug discovery process. It also addresses the implications of these technologies for clinical workflow, patient outcomes, and operational efficiency within healthcare institutions.",
  "Key Contributions": [
    "Analysis of machine learning models for improved disease prediction.",
    "Evaluation of AI tools facilitating early diagnosis of medical conditions.",
   

In [55]:
from azure.storage.blob import BlobServiceClient
AZURE_STORAGE_CONNECTION_STRING = os.getenv("endpoints")

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)
# container_name = "resume-output"
container_name = "himanicont"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print(" Upload successful")

 Upload successful


In [56]:
import datetime
file_name = f"research_summary_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

In [57]:
blob_client = blob_service_client.get_blob_client(
    container="himanicont",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F773859E450"',
 'last_modified': datetime.datetime(2026, 4, 21, 7, 25, 52, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'v\xb6\xae7\xe5\xe0rq\xb30\xfa\xc6\x8f\xc4&D'),
 'client_request_id': '5378769e-3d53-11f1-9830-db7f0d5fbf22',
 'request_id': '5a9efdbc-001e-0023-2460-d17451000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 7, 25, 52, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [58]:
print(f"Stored as: {file_name}")

Stored as: research_summary_20260421_125547.json


In [59]:
# Build & Deploy an AI Medical Report Interpreter (Azure AI Foundry)
# Objective
# Students will:
# Create AI resource
# Build an AI agent
# Deploy it
# Call it using Python (Colab)
# Store interpreted reports in cloud

In [60]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("key")

client = OpenAI(
    api_key=api_key, 
    base_url="https://him123.openai.azure.com/openai/v1"
)

In [61]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
report_data = """
Patient Name: John Doe
Age: 45
Test: Blood Report

Hemoglobin: 10 g/dL (Low)
WBC Count: 12000 cells/mcL (High)
Platelets: 150000 (Normal)

Doctor Notes: Possible infection and mild anemia.
"""

# 2. Use the Responses API for structured interpretation
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Interpret the medical report and return ONLY valid JSON with: Patient Name, Age, Summary, Abnormal Values, Possible Conditions, Recommendations."
                    },
                    {
                        "type": "input_text",
                        "text": f"MEDICAL REPORT:\n{report_data}"
                    }
                ]
            }
        ]
    )

    print("--- INTERPRETED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Interpretation Error: {e}")

--- INTERPRETED DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Patient Name": "John Doe",\n  "Age": 45,\n  "Summary": "Blood tests indicate low hemoglobin and elevated white blood cell count with normal platelets.",\n  "Abnormal Values": {\n    "Hemoglobin": "10 g/dL (Low)",\n    "WBC Count": "12000 cells/mcL (High)"\n  },\n  "Possible Conditions": [\n    "Mild anemia",\n    "Possible infection"\n  ],\n  "Recommendations": [\n    "Further evaluation for the source of infection",\n    "Dietary assessment and management for anemia",\n    "Repeat blood test as needed"\n  ]\n}', type='output_text', logprobs=[])]


In [62]:
import json

try:
    output_text_item = response.output[0].content[0]

    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    interpreted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(interpreted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Patient Name": "John Doe",
  "Age": 45,
  "Summary": "Blood tests indicate low hemoglobin and elevated white blood cell count with normal platelets.",
  "Abnormal Values": {
    "Hemoglobin": "10 g/dL (Low)",
    "WBC Count": "12000 cells/mcL (High)"
  },
  "Possible Conditions": [
    "Mild anemia",
    "Possible infection"
  ],
  "Recommendations": [
    "Further evaluation for the source of infection",
    "Dietary assessment and management for anemia",
    "Repeat blood test as needed"
  ]
}


In [63]:
import datetime
file_name = f"medical_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

In [64]:
from azure.storage.blob import BlobServiceClient

AZURE_STORAGE_CONNECTION_STRING = os.getenv("endpoints")
blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "himanicont"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=file_name
)

blob_client.upload_blob(
    json.dumps(interpreted_data, indent=2),
    overwrite=True
)

print("Upload successful")

Upload successful


In [65]:
print(f"Stored as: {file_name}")

Stored as: medical_report_20260421_125609.json
